In [2]:
import pandas as pd
import torch
import json
import re
import os
from datasets import Dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from peft import PeftModel
from tqdm import tqdm


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [1]:
!pip install unsloth xformers trl peft accelerate bitsandbytes


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.2/72.2 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.1/77.1 MB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 84.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 71.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 78.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 72.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 

In [3]:
torch.cuda.empty_cache()


In [4]:
train_df = pd.read_excel("/content/cases_train2 (4).xlsx")
test_df = pd.read_excel("/content/Test1_2 (2).xlsx")


In [5]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="meta-llama/Llama-3.1-8B-Instruct",
    max_seq_length=2048,
    load_in_4bit=True,
)

==((====))==  Unsloth 2026.7.6: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.1-8b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


In [6]:
def generate_critique(project_text, scores, model, tokenizer, max_new_tokens=1024):
    if pd.isna(project_text) or project_text == "":
        return "❌ Нет текста проекта"
    system_prompt = f"""Ты — эксперт по оценке бизнес-проектов.

Вот оценки проекта по 5 критериям (каждый от 1 до 5):
1. Анализ ЦА: {scores['ЦА']}/5
2. Проработка решения: {scores['Проработка']}/5
3. Финансовая модель: {scores['Финансы']}/5
4. Анализ рисков: {scores['Риски']}/5
5. Доказательства: {scores['Доказательства']}/5

Напиши РАЗВЕРНУТУЮ КРИТИКУ проекта.

ВАЖНО:
- Напиши УНИКАЛЬНУЮ критику для этого конкретного проекта
- Объясни, ПОЧЕМУ проект получил такие оценки
- Укажи, что можно улучшить
- Будь КОНКРЕТНЫМ, ссылайся на детали проекта
- НЕ КОПИРУЙ шаблонные фразы!
-Не используй смайлики
-Не используй слэнг
-Используй нейтральный стиль

Формат ответа:
1. Анализ ЦА (оценка X/5): [твой уникальный комментарий]
2. Проработка решения (оценка X/5): [твой уникальный комментарий]
3. Финансовая модель (оценка X/5): [твой уникальный комментарий]
4. Анализ рисков (оценка X/5): [твой уникальный комментарий]
5. Доказательства (оценка X/5): [твой уникальный комментарий]

Итоговая оценка: {sum(scores.values()) / 5:.1f}/5
Общий вывод: [твой уникальный вывод]
"""

    messages = [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": f"Вот текст проекта для оценки:\n\n{project_text[:3000]}"
        }
    ]

    try:
        inputs = tokenizer.apply_chat_template(
            messages,
            return_tensors="pt",
            add_generation_prompt=True
        ).to("cuda")

        with torch.no_grad():
            outputs = model.generate(
                inputs,
                max_new_tokens=max_new_tokens,
                temperature=0.8,
                do_sample=True,
                top_p=0.95,
                repetition_penalty=1.1,
            )
        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        if "assistant" in response:
            parts = response.split("assistant")
            if len(parts) > 1:
                return parts[-1].strip()
        return response
    except Exception as e:
        return f"❌ Ошибка: {str(e)}"


In [7]:
test_df["Критика_модели"] = None


In [8]:
for idx in tqdm(test_df.index, desc="Обработка кейсов"):
    project_text = test_df.loc[idx, "Решение кейса"]
    scores = {
        'ЦА': int(test_df.loc[idx, 'ЦА']) if 'ЦА' in test_df.columns else 3,
        'Проработка': int(test_df.loc[idx, 'Проработка решения']) if 'Проработка решения' in test_df.columns else 3,
        'Финансы': int(test_df.loc[idx, 'Финансовая модель и метрики']) if 'Финансовая модель и метрики' in test_df.columns else 3,
        'Риски': int(test_df.loc[idx, 'Анализ рисков']) if 'Анализ рисков' in test_df.columns else 3,
        'Доказательства': int(test_df.loc[idx, 'Доказательства']) if 'Доказательства' in test_df.columns else 3
    }
    critique = generate_critique(project_text, scores, model, tokenizer)
    test_df.loc[idx, "Критика_модели"] = critique
    if (idx + 1) % 5 == 0:
        test_df.to_excel("critique_partial.xlsx", index=False)
        print(f"\n💾 Сохранено {idx + 1} кейсов")


Обработка кейсов:   0%|          | 0/130 [00:00<?, ?it/s]The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=1024) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning


💾 Сохранено 5 кейсов


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Обработка кейсов:   5%|▍         | 6/130 [05:26<1:44:04, 50.36s/it]Both `max_new_tokens` (=1024) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transfo


💾 Сохранено 10 кейсов


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i

...


💾 Сохранено 30 кейсов


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Обработка кейсов:  24%|██▍       | 31/130 [27:30<1:26:14, 52.27s/it]Both `max_new_tokens` (=1024) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transf


💾 Сохранено 35 кейсов


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Обработка кейсов:  28%|██▊       | 36/130 [32:16<1:22:54, 52.92s/it]Both `max_new_tokens` (=1024) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transf

KeyboardInterrupt: 

In [ ]:
output_file = "test_with_critique.xlsx"
test_df.to_excel(output_file, index=False)